# Modelos GARCH Assimetricos: EGARCH e GJR-GARCH

Neste notebook, exploramos modelos que capturam o **efeito alavancagem** (leverage effect):
a observacao empirica de que choques negativos (quedas de preco) tendem a aumentar mais
a volatilidade do que choques positivos de mesma magnitude.

**Modelos cobertos:**
- EGARCH de Nelson (1991)
- GJR-GARCH de Glosten, Jagannathan e Runkle (1993)

**Conteudo:**
1. Motivacao: efeito alavancagem
2. EGARCH de Nelson (1991)
3. Interpretando o parametro de assimetria
4. GJR-GARCH de Glosten, Jagannathan e Runkle (1993)
5. News Impact Curve
6. Comparacao GARCH vs EGARCH vs GJR
7. Aplicacao: dados do Ibovespa

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd
from utils.plot_helpers import (
    plot_model_comparison,
)

from archbox.models import EGARCH, GARCH, GJRGARCH

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Motivacao: efeito alavancagem

O **efeito alavancagem** (leverage effect), documentado por Black (1976), refere-se a
correlacao negativa entre retornos e mudancas na volatilidade:

- Quando precos **caem**, a razao divida/patrimonio da empresa **aumenta** (maior alavancagem)
- Isso torna a empresa mais **arriscada**, elevando a volatilidade
- Choques **negativos** geram mais volatilidade que choques **positivos** de mesma magnitude

O GARCH(1,1) simetrico **nao captura** esse efeito — ele trata choques positivos e negativos
igualmente ($\epsilon_{t-1}^2$ ignora o sinal de $\epsilon_{t-1}$).

Vamos visualizar esse efeito nos dados.

In [ ]:
# Carregar dados do S&P 500
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

# Demonstrar correlacao negativa retorno-volatilidade
# Volatilidade proxy: retornos ao quadrado em janela movel
rolling_vol = returns.rolling(21).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(returns.index, returns.values, linewidth=0.5)
axes[0].set_title('Retornos S&P 500')
axes[0].set_ylabel('Retorno')

axes[1].plot(rolling_vol.index, rolling_vol.values, color='darkorange', linewidth=0.8)
axes[1].set_title('Volatilidade movel (21 dias)')
axes[1].set_ylabel('Desvio Padrao')

fig.tight_layout()
plt.show()

# Correlacao retornos vs mudanca na volatilidade
vol_change = rolling_vol.diff()
corr = returns.corr(vol_change)
print(f"Correlacao entre retornos e mudanca na volatilidade: {corr:.4f}")
print("(Valor negativo confirma o efeito alavancagem)")

## 2. EGARCH de Nelson (1991)

O modelo **EGARCH** (Exponential GARCH) modela o **logaritmo** da variancia condicional:

$$\ln(\sigma_t^2) = \omega + \alpha \left( |z_{t-1}| - E|z_{t-1}| \right) + \gamma z_{t-1} + \beta \ln(\sigma_{t-1}^2)$$

onde $z_t = \epsilon_t / \sigma_t$ sao os residuos padronizados.

**Vantagens do EGARCH:**
- Nao requer restricoes de nao-negatividade nos parametros (modela $\ln \sigma^2$)
- O parametro $\gamma$ captura a assimetria: se $\gamma < 0$, choques negativos aumentam mais a volatilidade
- Naturalmente garante $\sigma_t^2 > 0$

In [ ]:
# Estimate EGARCH(1,1)
model_egarch = EGARCH(returns.values, p=1, q=1)
results_egarch = model_egarch.fit()
print(results_egarch.summary())

## 3. Interpretando o parametro de assimetria $\gamma$

No EGARCH, o parametro $\gamma$ (gamma) mede a assimetria da resposta a choques:

- Se $\gamma = 0$: resposta simetrica (equivalente ao GARCH)
- Se $\gamma < 0$: choques negativos ($z_{t-1} < 0$) aumentam mais $\sigma_t^2$ → **efeito alavancagem**
- Se $\gamma > 0$: choques positivos aumentam mais $\sigma_t^2$ (raro em acoes)

Para um choque negativo ($z_{t-1} = -|z|$):
$$\text{impacto} = \alpha |z| - \gamma |z| = (\alpha - \gamma)|z|$$

Para um choque positivo ($z_{t-1} = |z|$):
$$\text{impacto} = \alpha |z| + \gamma |z| = (\alpha + \gamma)|z|$$

In [ ]:
# Extract and interpret the asymmetry parameter gamma
print("EGARCH Parameter Analysis")
print("=" * 50)

for name, param, pval in zip(results_egarch.param_names, results_egarch.params, results_egarch.pvalues, strict=False):
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
    print(f"{name:<15} = {param:>10.6f}  (p-value: {pval:.4f}) {sig}")

# Find gamma parameter
gamma_idx = [i for i, n in enumerate(results_egarch.param_names) if 'gamma' in n][0]
gamma_val = results_egarch.params[gamma_idx]
gamma_pval = results_egarch.pvalues[gamma_idx]

print(f"\nGamma = {gamma_val:.6f}")
print(f"Gamma p-value = {gamma_pval:.6f}")
if gamma_val < 0 and gamma_pval < 0.05:
    print("Gamma is negative and significant -> Leverage effect confirmed!")
    print("Negative shocks increase volatility more than positive shocks.")
elif gamma_val < 0:
    print("Gamma is negative but not statistically significant.")
else:
    print("Gamma is positive -> No standard leverage effect detected.")

## 4. GJR-GARCH de Glosten, Jagannathan e Runkle (1993)

O modelo **GJR-GARCH** adiciona um termo de **threshold** ao GARCH padrao:

$$\sigma_t^2 = \omega + (\alpha + \gamma I_{t-1}) \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

onde $I_{t-1}$ e uma funcao indicadora:
$$I_{t-1} = \begin{cases} 1 & \text{se } \epsilon_{t-1} < 0 \\ 0 & \text{caso contrario} \end{cases}$$

**Interpretacao:**
- Choque positivo: impacto = $\alpha \epsilon^2$
- Choque negativo: impacto = $(\alpha + \gamma) \epsilon^2$
- Se $\gamma > 0$: choques negativos tem impacto maior → **efeito alavancagem**

**Condicao de estacionariedade:** $\alpha + \beta + \gamma/2 < 1$

In [ ]:
# Estimate GJR-GARCH(1,1)
model_gjr = GJRGARCH(returns.values, p=1, q=1)
results_gjr = model_gjr.fit()
print(results_gjr.summary())

# Check leverage effect in GJR
gamma_idx_gjr = [i for i, n in enumerate(results_gjr.param_names) if 'gamma' in n][0]
gamma_gjr = results_gjr.params[gamma_idx_gjr]
gamma_pval_gjr = results_gjr.pvalues[gamma_idx_gjr]

print(f"\nGJR gamma = {gamma_gjr:.6f} (p-value: {gamma_pval_gjr:.6f})")
if gamma_gjr > 0 and gamma_pval_gjr < 0.05:
    print("Gamma > 0 and significant -> Leverage effect confirmed in GJR-GARCH!")
    print(f"Impact of negative shock: alpha + gamma = {results_gjr.params[1] + gamma_gjr:.6f}")
    print(f"Impact of positive shock: alpha = {results_gjr.params[1]:.6f}")
    print(f"Ratio (negative/positive): {(results_gjr.params[1] + gamma_gjr) / results_gjr.params[1]:.2f}x")

## 5. News Impact Curve

A **News Impact Curve** (NIC) mostra como choques passados ($\epsilon_{t-1}$) afetam a
variancia condicional corrente ($\sigma_t^2$), mantendo toda informacao anterior constante.

Para o GARCH simetrico, a NIC e uma **parabola centrada em zero** — choques positivos e
negativos de mesma magnitude geram o mesmo impacto na volatilidade.

Para modelos assimetricos (EGARCH, GJR), a NIC e **assimetrica** — choques negativos
geram maior impacto. Comparar as NICs dos diferentes modelos e uma forma visual
poderosa de entender as diferencas entre eles.

In [ ]:
# News Impact Curves: compare GARCH, EGARCH, and GJR-GARCH

# First estimate standard GARCH(1,1) as reference
model_garch = GARCH(returns.values, p=1, q=1)
results_garch = model_garch.fit(disp=False)

# Use archbox utility for news impact curves
from archbox.utils.news_impact import compare_news_impact, news_impact_curve

# Plot individual NICs
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (model, res, name) in zip(axes, [
    (model_garch, results_garch, 'GARCH(1,1)'),
    (model_egarch, results_egarch, 'EGARCH(1,1)'),
    (model_gjr, results_gjr, 'GJR-GARCH(1,1)'),
], strict=False):
    eps_range, sigma2_resp = news_impact_curve(model, res)
    ax.plot(eps_range, sigma2_resp, linewidth=1.5)
    ax.set_title(f'NIC - {name}')
    ax.set_xlabel(r'$\epsilon_{t-1}$')
    ax.set_ylabel(r'$\sigma^2_t$')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

fig.tight_layout()
plt.show()

# Overlay all three NICs on a single plot
fig, ax = plt.subplots(figsize=(10, 6))
models_results = [
    (model_garch, results_garch, 'GARCH(1,1)'),
    (model_egarch, results_egarch, 'EGARCH(1,1)'),
    (model_gjr, results_gjr, 'GJR-GARCH(1,1)'),
]
compare_news_impact(models_results, ax=ax)
ax.set_title('News Impact Curves - Model Comparison')
ax.legend()
fig.tight_layout()
plt.show()

print("The GARCH NIC is symmetric (parabola centered at zero).")
print("EGARCH and GJR NICs show asymmetry: steeper on the left (negative shocks).")
print("This visual clearly demonstrates the leverage effect captured by asymmetric models.")

## 6. Comparacao GARCH vs EGARCH vs GJR

Podemos comparar os modelos usando **criterios de informacao**:

| Criterio | Formula | Penalidade |
|----------|---------|------------|
| AIC | $-2\ell + 2k$ | Leve |
| BIC | $-2\ell + k \ln(n)$ | Moderada |
| HQIC | $-2\ell + 2k \ln(\ln(n))$ | Intermediaria |

onde $\ell$ e a log-verossimilhanca, $k$ o numero de parametros e $n$ o numero de observacoes.

**Menor valor = melhor modelo.** O BIC penaliza mais modelos complexos que o AIC.

In [ ]:
# Compare AIC/BIC/HQIC across the three models
comparison = pd.DataFrame({
    'GARCH': {
        'AIC': results_garch.aic, 'BIC': results_garch.bic, 'HQIC': results_garch.hqic,
        'LogLik': results_garch.loglike, 'Params': len(results_garch.params),
    },
    'EGARCH': {
        'AIC': results_egarch.aic, 'BIC': results_egarch.bic, 'HQIC': results_egarch.hqic,
        'LogLik': results_egarch.loglike, 'Params': len(results_egarch.params),
    },
    'GJR-GARCH': {
        'AIC': results_gjr.aic, 'BIC': results_gjr.bic, 'HQIC': results_gjr.hqic,
        'LogLik': results_gjr.loglike, 'Params': len(results_gjr.params),
    },
})
print("Information Criteria Comparison")
print("=" * 60)
print(comparison.T.to_string())

# Identify best model per criterion
for criterion in ['AIC', 'BIC', 'HQIC']:
    best = comparison.loc[criterion].astype(float).idxmin()
    print(f"\nBest by {criterion}: {best} ({comparison.loc[criterion, best]:.4f})")

# Visualize with bar chart
results_dict = {'GARCH': results_garch, 'EGARCH': results_egarch, 'GJR-GARCH': results_gjr}
plot_model_comparison(results_dict)
plt.show()

## 7. Aplicacao: dados do Ibovespa

Agora aplique os tres modelos nos dados do **Ibovespa** (indice da bolsa brasileira).

Mercados emergentes como o Brasil tendem a ter:
- Maior volatilidade base
- Efeito alavancagem potencialmente mais forte
- Caudas mais pesadas

Compare os resultados com os do S&P 500.

In [ ]:
# Application: Ibovespa data
ibov = pd.read_csv('../data/ibovespa_returns.csv', parse_dates=['date'], index_col='date')
ibov_returns = ibov['returns']

print(f"Ibovespa dataset: {len(ibov_returns)} observations")
print(f"Period: {ibov_returns.index[0].date()} to {ibov_returns.index[-1].date()}")
print(f"Mean: {ibov_returns.mean():.6f}, Std: {ibov_returns.std():.6f}\n")

# Estimate all three models on Ibovespa
model_garch_ibov = GARCH(ibov_returns.values, p=1, q=1)
results_garch_ibov = model_garch_ibov.fit(disp=False)

model_egarch_ibov = EGARCH(ibov_returns.values, p=1, q=1)
results_egarch_ibov = model_egarch_ibov.fit(disp=False)

model_gjr_ibov = GJRGARCH(ibov_returns.values, p=1, q=1)
results_gjr_ibov = model_gjr_ibov.fit(disp=False)

# Parameter comparison: S&P 500 vs Ibovespa
print("Parameter Comparison: S&P 500 vs Ibovespa")
print("=" * 70)

for name, sp_res, ibov_res in [
    ('GARCH', results_garch, results_garch_ibov),
    ('EGARCH', results_egarch, results_egarch_ibov),
    ('GJR-GARCH', results_gjr, results_gjr_ibov),
]:
    print(f"\n--- {name} ---")
    for pname, sp_val, ibov_val in zip(sp_res.param_names, sp_res.params, ibov_res.params, strict=False):
        print(f"  {pname:<15} S&P500: {sp_val:>10.6f}  Ibovespa: {ibov_val:>10.6f}")

# Information criteria comparison for Ibovespa
ibov_comparison = pd.DataFrame({
    'GARCH': {'AIC': results_garch_ibov.aic, 'BIC': results_garch_ibov.bic},
    'EGARCH': {'AIC': results_egarch_ibov.aic, 'BIC': results_egarch_ibov.bic},
    'GJR-GARCH': {'AIC': results_gjr_ibov.aic, 'BIC': results_gjr_ibov.bic},
})
print("\n\nIbovespa Information Criteria")
print("=" * 50)
print(ibov_comparison.T.to_string())

# Volatility comparison plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(results_garch.conditional_volatility, linewidth=0.8, label='GARCH')
axes[0].plot(results_egarch.conditional_volatility, linewidth=0.8, label='EGARCH')
axes[0].plot(results_gjr.conditional_volatility, linewidth=0.8, label='GJR-GARCH')
axes[0].set_title('S&P 500 - Conditional Volatility')
axes[0].legend()

axes[1].plot(results_garch_ibov.conditional_volatility, linewidth=0.8, label='GARCH')
axes[1].plot(results_egarch_ibov.conditional_volatility, linewidth=0.8, label='EGARCH')
axes[1].plot(results_gjr_ibov.conditional_volatility, linewidth=0.8, label='GJR-GARCH')
axes[1].set_title('Ibovespa - Conditional Volatility')
axes[1].legend()

fig.tight_layout()
plt.show()

## Conclusao

Neste notebook, aprendemos:

- O **efeito alavancagem** e por que modelos simetricos sao insuficientes
- O modelo **EGARCH**: modela $\ln(\sigma^2_t)$, parametro $\gamma$ captura assimetria
- O modelo **GJR-GARCH**: usa indicadora $I_{t-1}$ para diferenciar choques negativos
- A **News Impact Curve** como ferramenta visual para comparar modelos
- Como usar **criterios de informacao** para selecao de modelos

No proximo notebook, exploraremos o **APARCH** (potencia variavel) e o
**Component-GARCH** (decomposicao em componentes de curto e longo prazo).